# Caso de Estudio: Análisis Exploratorio de las Planillas del Mundial 2026
## Fase 1 y 2 del Proceso de Google: Formular Preguntas y Preparar los Datos

### Contexto del Proyecto
Con la expansión del Mundial 2026 a 48 selecciones, se han convocado a un récord de 1,248 jugadores. Este proyecto tiene como objetivo extraer, limpiar y analizar los datos de los planteles justo antes del inicio del torneo para descubrir patrones demográficos, la representación de las ligas de fútbol globales y el nivel de competitividad de cada selección.

### Objetivos de Negocio / Preguntas Clave:
1. ¿Cuál es el perfil de edad y experiencia de las selecciones participantes?
2. ¿Qué ligas y clubes del mundo aportan más jugadores al torneo?
3. ¿Qué selecciones dependen más de jugadores que militan en el extranjero?

### Estrategia de Extracción:
Dado que los datos están en constante actualización, utilizaremos técnicas de **Web Scraping** en Python con `BeautifulSoup` para extraer las tablas oficiales directamente desde Wikipedia.

In [1]:
# Importamos las librerías necesarias para el proyecto
import requests
from bs4 import BeautifulSoup
import pandas as pd

print("Librerías cargadas exitosamente.")

Librerías cargadas exitosamente.


### Extracción Automatizada de Datos (Web Scraping)
A continuación, definimos la URL de Wikipedia que contiene las tablas de las 48 selecciones. Para evitar que los servidores bloqueen la petición automatizada, configuramos un `User-Agent` que simula la navegación de un usuario real en Google Chrome.

In [2]:
# 1. URL oficial de los planteles en Wikipedia
url = "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_squads"

# 2. Configuración del encabezado para evitar bloqueos
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

print("Conectando con Wikipedia...")
response = requests.get(url, headers=headers)

# 3. Procesar el HTML
soup = BeautifulSoup(response.content, 'lxml')

# 4. Buscar las tablas de los planteles
tablas = soup.find_all('table', class_='wikitable')
print(f"Se encontraron {len(tablas)} tablas en la página. Extrayendo datos...")

lista_jugadores = []

# 5. Recorrer las tablas y extraer filas
for index, tabla in enumerate(tablas):
    filas = tabla.find_all('tr')
    
    for fila in filas[1:]:
        celdas = fila.find_all(['td', 'th'])
        
        if len(celdas) >= 6:
            try:
                numero = celdas[0].text.strip()
                posicion = celdas[1].text.strip()
                nombre = celdas[2].text.strip()
                edad = celdas[3].text.strip()
                club = celdas[-1].text.strip()
                
                lista_jugadores.append({
                    "Tabla_Index": index,
                    "Numero": numero,
                    "Posicion": posicion,
                    "Nombre": nombre,
                    "Edad_Fecha": edad,
                    "Club": club
                })
            except Exception as e:
                continue

# 6. Crear el DataFrame y confirmar resultados
df_crudo = pd.DataFrame(lista_jugadores)
print(f"¡Éxito! Se han extraído {len(df_crudo)} registros de jugadores.")

Conectando con Wikipedia...
Se encontraron 53 tablas en la página. Extrayendo datos...
¡Éxito! Se han extraído 1248 registros de jugadores.


### Inspección Inicial del Dataset (Fase de Diagnóstico)
Una vez guardados los datos en memoria, generamos una vista previa del DataFrame para identificar la estructura y definir las tareas de limpieza necesarias para la siguiente fase.

In [3]:
# Visualizar las primeras 5 filas del dataset
df_crudo.head()

,Tabla_Index,Numero,Posicion,Nombre,Edad_Fecha,Club
0,0,1,1GK,Matěj Kovář,"(2000-05-17)May 17, 2000 (aged 26)",PSV Eindhoven
1,0,2,2DF,David Zima,"(2000-11-08)November 8, 2000 (aged 25)",Slavia Prague
2,0,3,2DF,Tomáš Holeš,"(1993-03-31)March 31, 1993 (aged 33)",Slavia Prague
3,0,4,2DF,Robin Hranáč,"(2000-01-29)January 29, 2000 (aged 26)",TSG Hoffenheim
4,0,5,2DF,Vladimír Coufal,"(1992-08-22)August 22, 1992 (aged 33)",TSG Hoffenheim


In [4]:
# Guardamos los datos crudos en un archivo CSV independiente para mantener el histórico
df_crudo.to_csv("datos_crudos_mundial.csv", index=False)
print("Archivo 'datos_crudos_mundial.csv' guardado correctamente.")

Archivo 'datos_crudos_mundial.csv' guardado correctamente.


### Próximos Pasos: Fase de Procesamiento (Limpieza de Datos)
Al observar la vista previa de `df_crudo`, identificamos los siguientes problemas de calidad de datos que resolveremos en el próximo Notebook mediante Pandas/SQL:
1. **Columna Nombre:** Contiene texto extra como "(captain)" que debe ser removido.
2. **Columna Edad_Fecha:** Mezcla la fecha en formato string y la edad entre paréntesis. Necesitamos separar esto en columnas numéricas limpias.
3. **Estandarización:** Verificar caracteres especiales en nombres de jugadores y clubes.